# EDA: DJ Mixing Station Studio (Combined Dataset)

This notebook explores the current **combined offline-v2 dataset** built from:
- `salvatorerastelli/spotify-and-youtube` (baseline)
- `solomonameh/spotify-music-dataset` (expansion)

Focus areas:
1. Data quality and schema checks
2. Distribution and correlation analysis
3. Artist/track coverage and popularity patterns
4. Duplicate/link integrity checks
5. Modeling-based signal analysis with **Plotly + Matplotlib/Seaborn + SHAP (optional)**

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

# Optional SHAP
try:
    import shap  # type: ignore
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

# Better Matplotlib cache location in local project
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.mplconfig'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

In [ ]:
# Resolve project code path from notebook folder
nb_dir = Path.cwd()
code_dir = (nb_dir.parent / 'code').resolve()
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

import data_upgrade_v2
import recommender_v2_adapter

raw_df, source_path, dataset_path = data_upgrade_v2.load_and_merge_from_kagglehub()
prepared_df = recommender_v2_adapter.prepare_music_data(raw_df)

print('Source CSV path(s):', source_path)
print('Dataset cache path(s):', dataset_path)
print('Raw shape:', raw_df.shape)
print('Prepared shape:', prepared_df.shape)

## 1. Schema and Data Quality Overview

In [ ]:
def _safe_nunique(series: pd.Series) -> int:
    def _to_hashable(v):
        if isinstance(v, list):
            return tuple(v)
        if isinstance(v, dict):
            return tuple(sorted(v.items()))
        return v
    return pd.Series([_to_hashable(v) for v in series]).nunique(dropna=True)

summary = pd.DataFrame({
    'column': prepared_df.columns,
    'dtype': [str(prepared_df[c].dtype) for c in prepared_df.columns],
    'missing_pct': [prepared_df[c].isna().mean() * 100 for c in prepared_df.columns],
    'n_unique': [_safe_nunique(prepared_df[c]) for c in prepared_df.columns],
}).sort_values(['missing_pct', 'n_unique'], ascending=[False, False])

display(summary.head(40))

In [ ]:
key_checks = {
    'rows_prepared': len(prepared_df),
    'artists_unique': prepared_df['artist'].nunique(),
    'tracks_unique_text': prepared_df['track'].nunique(),
    'canonical_song_key_unique': prepared_df['canonical_song_key'].nunique() if 'canonical_song_key' in prepared_df.columns else np.nan,
    'has_spotify_track_id_pct': (prepared_df['spotify_track_id'].fillna('').astype(str).str.len() > 0).mean() * 100 if 'spotify_track_id' in prepared_df.columns else np.nan,
    'has_youtube_video_id_pct': (prepared_df['youtube_video_id'].fillna('').astype(str).str.len() > 0).mean() * 100 if 'youtube_video_id' in prepared_df.columns else np.nan,
}
pd.Series(key_checks)

In [ ]:
# Missingness visualization (top 30 columns by missingness)
miss = prepared_df.isna().mean().sort_values(ascending=False).head(30)
plt.figure(figsize=(12, 6))
sns.barplot(x=miss.values * 100, y=miss.index, color='#c9a227')
plt.title('Top 30 Columns by Missing %')
plt.xlabel('Missing (%)')
plt.ylabel('Column')
plt.tight_layout()
plt.show()

## 2. Numeric Distributions and Correlations

In [ ]:
num_cols = [
    'views', 'stream', 'likes', 'comments', 'duration_ms',
    'danceability', 'energy', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
    'momentum_score', 'popularity_norm', 'engagement_rate', 'stream_to_view_ratio'
]
num_cols = [c for c in num_cols if c in prepared_df.columns]

prepared_num = prepared_df[num_cols].copy()
prepared_num = prepared_num.replace([np.inf, -np.inf], np.nan)

In [ ]:
for c in ['views', 'stream', 'likes', 'comments']:
    if c in prepared_num.columns:
        prepared_num[f'log1p_{c}'] = np.log1p(pd.to_numeric(prepared_num[c], errors='coerce').fillna(0))

plot_cols = [c for c in ['log1p_views', 'log1p_stream', 'log1p_likes', 'log1p_comments', 'momentum_score', 'danceability', 'energy', 'valence'] if c in prepared_num.columns]

for col in plot_cols:
    fig = px.histogram(prepared_num, x=col, nbins=60, title=f'Distribution: {col}', template='plotly_dark')
    fig.update_layout(height=380)
    fig.show()

In [ ]:
corr_cols = [c for c in [
    'danceability', 'energy', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'speechiness', 'tempo_scaled',
    'momentum_score', 'popularity_norm', 'stream_norm', 'views_norm'
] if c in prepared_df.columns]

corr = prepared_df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, square=True)
plt.title('Correlation Heatmap (Core Features + Signals)')
plt.tight_layout()
plt.show()

## 3. Coverage by Source and Tier

In [ ]:
if 'source_dataset' in prepared_df.columns:
    source_counts = prepared_df['source_dataset'].value_counts(dropna=False).reset_index()
    source_counts.columns = ['source_dataset', 'count']
    fig = px.bar(source_counts, x='source_dataset', y='count', title='Rows by Source Dataset', color='source_dataset', template='plotly_dark')
    fig.show()

if 'source_tier' in prepared_df.columns:
    tier_counts = prepared_df['source_tier'].value_counts(dropna=False).reset_index()
    tier_counts.columns = ['source_tier', 'count']
    fig = px.bar(tier_counts, x='source_tier', y='count', title='Rows by Source Tier', color='source_tier', template='plotly_dark')
    fig.show()

## 4. Top Artists and Tracks (Engagement/Popularity)

In [ ]:
artist_agg = prepared_df.groupby('artist', as_index=False).agg(
    tracks=('track', 'count'),
    total_stream=('stream', 'sum'),
    total_views=('views', 'sum'),
    avg_momentum=('momentum_score', 'mean'),
)
artist_agg['artist_score'] = np.log1p(artist_agg['total_stream']) + np.log1p(artist_agg['total_views']) + 2 * artist_agg['avg_momentum']

top_artists = artist_agg.sort_values('artist_score', ascending=False).head(20)
fig = px.bar(top_artists.sort_values('artist_score'), x='artist_score', y='artist', orientation='h',
             title='Top 20 Artists by Composite Score', template='plotly_dark')
fig.update_layout(height=700)
fig.show()

In [ ]:
track_agg = prepared_df[['artist', 'track', 'stream', 'views', 'momentum_score']].copy()
track_agg['track_label'] = track_agg['artist'] + ' - ' + track_agg['track']
track_agg['track_score'] = np.log1p(track_agg['stream']) + np.log1p(track_agg['views']) + 2 * track_agg['momentum_score']

top_tracks = track_agg.sort_values('track_score', ascending=False).head(20)
fig = px.bar(top_tracks.sort_values('track_score'), x='track_score', y='track_label', orientation='h',
             title='Top 20 Tracks by Composite Score', template='plotly_dark')
fig.update_layout(height=760)
fig.show()

## 5. Duplicate and Link Integrity Audit

In [ ]:
audit = prepared_df.copy()

if 'spotify_track_id' in audit.columns:
    sp_nonempty = audit['spotify_track_id'].fillna('').astype(str).str.strip() != ''
    sp_dup_count = audit.loc[sp_nonempty, 'spotify_track_id'].duplicated().sum()
else:
    sp_dup_count = np.nan

if 'youtube_video_id' in audit.columns:
    yt_nonempty = audit['youtube_video_id'].fillna('').astype(str).str.strip() != ''
    yt_dup_count = audit.loc[yt_nonempty, 'youtube_video_id'].duplicated().sum()
else:
    yt_dup_count = np.nan

no_direct_youtube = (audit['youtube_link'].fillna('').astype(str).str.contains(r'(?:watch\?v=|youtu\.be/)') == False).sum() if 'youtube_link' in audit.columns else np.nan

pd.Series({
    'spotify_track_id_duplicates': int(sp_dup_count) if pd.notna(sp_dup_count) else np.nan,
    'youtube_video_id_duplicates': int(yt_dup_count) if pd.notna(yt_dup_count) else np.nan,
    'no_direct_youtube_rows': int(no_direct_youtube) if pd.notna(no_direct_youtube) else np.nan,
})

In [ ]:
# show sample rows without direct YouTube IDs
sample_no_direct = prepared_df.loc[
    ~prepared_df['youtube_link'].fillna('').astype(str).str.contains(r'(?:watch\?v=|youtu\.be/)'),
    ['artist', 'track', 'youtube_link', 'url_youtube']
].drop_duplicates(subset=['artist', 'track']).head(20)

display(sample_no_direct)

## 6. Lightweight Predictive Modeling (Feature Importance)

In [ ]:
model_df = prepared_df.copy()
feature_cols = [c for c in [
    'danceability', 'energy', 'acousticness', 'instrumentalness', 'liveness',
    'valence', 'speechiness', 'tempo_scaled',
    'views_norm', 'stream_norm', 'engagement_norm', 'ratio_norm', 'popularity_norm'
] if c in model_df.columns]

target_col = 'momentum_score'
model_data = model_df[feature_cols + [target_col]].replace([np.inf, -np.inf], np.nan).dropna()

X = model_data[feature_cols]
y = model_data[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

rf = RandomForestRegressor(
    n_estimators=250,
    max_depth=18,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
print('R2:', round(r2_score(y_test, pred), 4))
print('MAE:', round(mean_absolute_error(y_test, pred), 4))

In [ ]:
# Permutation importance (always available)
perm = permutation_importance(rf, X_test, y_test, n_repeats=8, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)

display(imp_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=imp_df.head(15), x='importance_mean', y='feature', color='#c9a227')
plt.title('Permutation Importance (RandomForest)')
plt.xlabel('Mean Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 7. SHAP Analysis (Optional)

In [ ]:
# !pip install shap

In [ ]:
if SHAP_AVAILABLE:
    sample_n = min(1200, len(X_train))
    shap_sample = X_train.sample(sample_n, random_state=42)

    # TreeExplainer is efficient for RandomForest
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(shap_sample)

    # Summary beeswarm
    shap.summary_plot(shap_values, shap_sample, max_display=20)

    # Bar summary
    shap.summary_plot(shap_values, shap_sample, plot_type='bar', max_display=20)
else:
    print('SHAP is not installed in this environment. Install it with: pip install shap')
    print('Permutation importance above is used as the fallback global attribution view.')

## 8. Recommendation-Oriented Quick Checks

These checks align EDA with app behavior:
- audio features are well-populated for similarity scoring
- momentum/popularity signals are distributed and non-degenerate
- link fields have measurable direct/indirect coverage
- duplicate identity leakage can be audited before ranking

In [ ]:
quick_checks = {
    'feature_non_null_pct': float(prepared_df[[c for c in ['danceability','energy','acousticness','instrumentalness','liveness','valence','speechiness','tempo_scaled'] if c in prepared_df.columns]].notna().mean().mean() * 100),
    'momentum_std': float(prepared_df['momentum_score'].std()),
    'popularity_std': float(prepared_df['popularity_norm'].std()),
    'direct_spotify_pct': float((prepared_df['spotify_track_id'].fillna('').astype(str).str.len() > 0).mean() * 100),
    'direct_youtube_pct': float((prepared_df['youtube_video_id'].fillna('').astype(str).str.len() > 0).mean() * 100),
}

pd.Series(quick_checks)